In [1]:
import csv
import os
import cv2
import mediapipe as mp
import numpy as np
import requests
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Automatic File & Folder Setup
DATASET_DIR = "dataset"
MODEL_DIR = os.path.join("model", "keypoint_classifier")
CSV_PATH = os.path.join(DATASET_DIR, "keypoint.csv")
MODEL_PATH = os.path.join(MODEL_DIR, "hand_landmarker.task")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Auto-download the Google Model if you don't have it yet
if not os.path.exists(MODEL_PATH):
    print("📥 Downloading hand_landmarker.task from Google servers... please wait...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    response = requests.get(url)
    with open(MODEL_PATH, "wb") as f:
        f.write(response.content)
    print("✅ Download Complete!")

# 2. Initialize the Modern MediaPipe Landmarker
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.5
)
landmarker = vision.HandLandmarker.create_from_options(options)

# 3. Math Normalization Helper
def normalize_landmarks(hand_landmarks_list):
    landmark_list = []
    for lm in hand_landmarks_list:
        landmark_list.append([lm.x, lm.y])
        
    np_landmarks = np.array(landmark_list)
    wrist = np_landmarks[0]
    normalized = np_landmarks - wrist # Translate relative to wrist
    flattened = normalized.flatten().tolist()
    
    # Scale boundaries between -1 and 1
    max_val = max(max(abs(val) for val in flattened), 1e-6)
    flattened = [val / max_val for val in flattened]
    return flattened

def log_to_csv(label, flattened_coords):
    file_exists = os.path.isfile(CSV_PATH)
    with open(CSV_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            headers = ['label'] + [f'pt{i}_{axis}' for i in range(21) for axis in ['x', 'y']]
            writer.writerow(headers)
        writer.writerow([label] + flattened_coords)

# 4. Interactive Live Camera Loop
cap = cv2.VideoCapture(0)
print("\n🚀 === SYSTEM WORKING PERFECTLY ===")
print("• Hold up a hand gesture.")
print("• Press keys 0-4 to save data (0: Palm, 1: Fist, 2: Thumbs Up, 4: OK Sign).")
print("• Press 'q' to close the program.\n")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Camera feed missing.")
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Process Frame
    results = landmarker.detect(mp_image)
    current_landmarks = None

    if results.hand_landmarks:
        current_landmarks = results.hand_landmarks[0]
        
        # Draw on-screen tracker skeleton manually
        h, w, _ = frame.shape
        for lm in current_landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    # Key Listeners
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif ord('0') <= key <= ord('9') and current_landmarks is not None:
        label = chr(key)
        normalized_data = normalize_landmarks(current_landmarks)
        log_to_csv(label, normalized_data)
        print(f"Recorded sample for Gesture ID: {label}")
        cv2.putText(frame, f"SAVED {label}", (50, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.putText(frame, "Keys 0-9 to Log | 'q' to Quit", (10, frame.shape[0] - 15), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.imshow('Gesture Dataset Builder', frame)

cap.release()
cv2.destroyAllWindows()


🚀 === SYSTEM WORKING PERFECTLY ===
• Hold up a hand gesture.
• Press keys 0-4 to save data (0: Palm, 1: Fist, 2: Thumbs Up, 4: OK Sign).
• Press 'q' to close the program.

Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sample for Gesture ID: 0
Recorded sa